# Hy-BiSSM — Hyperspectral Classification on Indian Pines
**Indian Pines Dataset | 200 Bands | 145×145 pixels | 16 Classes**

Hy-BiSSM fuses a Bidirectional SSM spectral encoder with a Dual-Scale Spatial Context module.
Training uses only the cross-entropy loss (L_CE) for straightforward end-to-end optimisation.

```
Architecture overview:
  M1 — BidirectionalSSM  : processes the spectral sequence per central pixel
  M2 — DualScaleSpatialContext : local (7×7) + global (25×25) patch branches with a scalar gate
  Classifier  : fused spectral + spatial features → 16 class logits
```

## 0. Environment

In [ ]:
!pip install torch --quiet
!pip install ninja packaging --quiet
!pip install seaborn scipy --quiet
import subprocess, sys
try:
    import mamba_ssm
    print('mamba_ssm already installed')
except ImportError:
    print('Installing mamba-ssm...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'mamba-ssm', '--no-build-isolation', '--quiet'])

In [ ]:
import os, random, time, warnings
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print(f'Device : {DEVICE}')
print(f'GPUs   : {N_GPUS}')
for i in range(N_GPUS):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

## 1. Configuration

In [ ]:
class CFG:
    # ── Data ──────────────────────────────────────────────────────────────────
    # Path to the Indian Pines .mat files.
    # Expected files:
    #   Indian_pines_corrected.mat  → key 'indian_pines_corrected'  (145,145,200)
    #   Indian_pines_gt.mat         → key 'indian_pines_gt'          (145,145)
    DATA_ROOT    = 'Indian_Pines'
    IMAGE_FILE   = 'Indian_pines_corrected.mat'
    LABEL_FILE   = 'Indian_pines_gt.mat'

    N_CLASSES    = 16
    N_BANDS      = 200
    CLASS_NAMES  = [
        'Alfalfa', 'Corn-notill', 'Corn-mintill', 'Corn',
        'Grass-pasture', 'Grass-trees', 'Grass-pasture-mowed',
        'Hay-windrowed', 'Oats', 'Soybean-notill', 'Soybean-mintill',
        'Soybean-clean', 'Wheat', 'Woods',
        'Buildings-Grass-Trees-Drives', 'Stone-Steel-Towers'
    ]

    # ── Patch sizes ───────────────────────────────────────────────────────────
    LOCAL_SIZE   = 7
    GLOBAL_SIZE  = 25
    LOCAL_PAD    = LOCAL_SIZE  // 2
    GLOBAL_PAD   = GLOBAL_SIZE // 2

    # ── Training ──────────────────────────────────────────────────────────────
    TRAIN_RATIO  = 0.10    # 10% of labeled pixels for training
    BATCH_SIZE   = 64
    EPOCHS       = 150
    LR           = 3e-4
    WEIGHT_DECAY = 1e-4
    PATIENCE     = 20

    # ── Model dims ────────────────────────────────────────────────────────────
    SPEC_DIM     = 64
    SPAT_DIM     = 128
    FUSED_DIM    = SPEC_DIM + SPAT_DIM   # 192

    # ── Output ────────────────────────────────────────────────────────────────
    SAVE_PATH    = 'output/hybissm_indian_pines.pt'

## 2. Data Loading — Indian Pines

In [ ]:
def load_indian_pines(cfg):
    """
    Loads the Indian Pines hyperspectral dataset.
    Returns:
        image  : (145, 145, 200) float32, per-band normalised to [0, 1]
        labels : (145, 145)      int64,   0 = unlabelled, 1-16 = class
    """
    data_dir = Path(cfg.DATA_ROOT)

    img_path = data_dir / cfg.IMAGE_FILE
    lbl_path = data_dir / cfg.LABEL_FILE

    # ── Load image ────────────────────────────────────────────────────────────
    img_mat = sio.loadmat(str(img_path))
    img_key = [k for k in img_mat if not k.startswith('_')][0]
    image   = img_mat[img_key].astype(np.float32)   # (145, 145, 200)

    # Ensure shape is (H, W, B)
    if image.ndim == 3 and image.shape[0] == cfg.N_BANDS:
        image = image.transpose(1, 2, 0)

    # Per-band min-max normalisation
    for b in range(image.shape[2]):
        mn, mx = image[:, :, b].min(), image[:, :, b].max()
        if mx > mn:
            image[:, :, b] = (image[:, :, b] - mn) / (mx - mn)

    # ── Load ground-truth labels ──────────────────────────────────────────────
    lbl_mat = sio.loadmat(str(lbl_path))
    lbl_key = [k for k in lbl_mat if not k.startswith('_')][0]
    labels  = lbl_mat[lbl_key].astype(np.int64).squeeze()  # (145, 145)

    n_labeled = (labels > 0).sum()
    print(f'Image shape  : {image.shape}')
    print(f'Labels shape : {labels.shape}')
    print(f'Labeled pixels: {n_labeled:,}  (background: {(labels==0).sum():,})')

    # Per-class pixel count
    print('\nClass distribution:')
    for c in range(1, cfg.N_CLASSES + 1):
        cnt = (labels == c).sum()
        print(f'  [{c:>2}] {cfg.CLASS_NAMES[c-1]:<35s}: {cnt:>5,}')

    return image, labels


os.makedirs('output', exist_ok=True)
image, labels = load_indian_pines(CFG)

## 3. Dataset — Dual-Patch

In [ ]:
class DualPatchDataset(Dataset):
    """
    Returns (local_patch, global_patch, label) per labeled pixel.
    local_patch  : (N_BANDS, LOCAL_SIZE, LOCAL_SIZE)
    global_patch : (N_BANDS, GLOBAL_SIZE, GLOBAL_SIZE)
    label        : int, 0-indexed
    Padding: reflect, computed once at init.
    """
    def __init__(self, image, labels, cfg, indices, augment=False):
        self.cfg     = cfg
        self.labels  = labels
        self.indices = indices
        self.augment = augment

        pad = cfg.GLOBAL_PAD
        self.padded = np.pad(image,
                             ((pad, pad), (pad, pad), (0, 0)),
                             mode='reflect')
        self.offset = pad

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        r, c = self.indices[idx]
        ro   = r + self.offset
        co   = c + self.offset
        lp   = self.cfg.LOCAL_PAD
        gp   = self.cfg.GLOBAL_PAD

        local_p  = self.padded[ro-lp:ro+lp+1, co-lp:co+lp+1, :]
        global_p = self.padded[ro-gp:ro+gp+1, co-gp:co+gp+1, :]

        local_p  = torch.from_numpy(local_p.transpose(2, 0, 1).copy()).float()
        global_p = torch.from_numpy(global_p.transpose(2, 0, 1).copy()).float()

        if self.augment:
            if random.random() > 0.5:
                local_p  = torch.flip(local_p,  dims=[2])
                global_p = torch.flip(global_p, dims=[2])
            if random.random() > 0.5:
                local_p  = torch.flip(local_p,  dims=[1])
                global_p = torch.flip(global_p, dims=[1])

        label = int(self.labels[r, c]) - 1   # 0-indexed
        return local_p, global_p, label


def make_splits(labels, cfg):
    """Returns (train_idx, val_idx, test_idx) as lists of (row, col)."""
    rows, cols = np.where(labels > 0)
    all_idx    = list(zip(rows.tolist(), cols.tolist()))
    all_lbl    = [labels[r, c] for r, c in all_idx]

    train_idx, temp_idx, _, temp_lbl = train_test_split(
        all_idx, all_lbl,
        train_size=cfg.TRAIN_RATIO,
        stratify=all_lbl, random_state=SEED)

    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5,
        stratify=temp_lbl, random_state=SEED)

    return train_idx, val_idx, test_idx

In [ ]:
# Build train / val / test splits
train_idx, val_idx, test_idx = make_splits(labels, CFG)

train_ds = DualPatchDataset(image, labels, CFG, train_idx, augment=True)
val_ds   = DualPatchDataset(image, labels, CFG, val_idx,   augment=False)
test_ds  = DualPatchDataset(image, labels, CFG, test_idx,  augment=False)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')
print(f'Steps per epoch: {len(train_loader):,}')

## 4. Model — Hy-BiSSM

In [ ]:
# ── Module 1: Spectral Encoder (BidirectionalSSM) ─────────────────────────────

class BidirectionalSSM(nn.Module):
    def __init__(self, n_bands, d_model=32):
        super().__init__()
        self.d_model   = d_model
        self.use_mamba = False
        try:
            from mamba_ssm import Mamba
            self.fwd_mamba = Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2)
            self.bwd_mamba = Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2)
            self.proj_in   = nn.Linear(1, d_model)
            self.use_mamba = True
            print('BidirectionalSSM: mamba_ssm.Mamba')
        except ImportError:
            self.bigru = nn.GRU(input_size=1, hidden_size=d_model,
                                num_layers=2, batch_first=True,
                                bidirectional=True, dropout=0.1)
            print('BidirectionalSSM: BiGRU fallback')

    def forward(self, x):
        seq = x.unsqueeze(-1)                        # (B, n_bands, 1)
        if self.use_mamba:
            seq = self.proj_in(seq)                  # (B, n_bands, d_model)
            fwd = self.fwd_mamba(seq)
            bwd = self.bwd_mamba(seq.flip(1)).flip(1)
            return torch.cat([fwd, bwd], dim=-1).mean(dim=1)  # (B, d_model*2)
        else:
            out, _ = self.bigru(seq)
            return out.mean(dim=1)                   # (B, d_model*2)


class SpectralEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        ssm_dim  = 32
        self.ssm = BidirectionalSSM(cfg.N_BANDS, d_model=ssm_dim)
        self.proj = nn.Sequential(
            nn.Linear(ssm_dim * 2, cfg.SPEC_DIM),
            nn.LayerNorm(cfg.SPEC_DIM), nn.GELU()
        )

    def forward(self, x):
        return self.proj(self.ssm(x))   # (B, SPEC_DIM)

In [ ]:
# ── Module 2: Dual-Scale Spatial Context ─────────────────────────────────────

class SpatialBranch(nn.Module):
    def __init__(self, n_bands, out_dim, patch_size):
        super().__init__()
        self.conv3d = nn.Sequential(
            nn.Conv3d(1,  8, kernel_size=(7, 3, 3), padding=(3, 1, 1)),
            nn.BatchNorm3d(8),  nn.GELU(),
            nn.Conv3d(8, 16, kernel_size=(5, 3, 3), padding=(2, 1, 1)),
            nn.BatchNorm3d(16), nn.GELU(),
            nn.Conv3d(16, 32, kernel_size=(3, 3, 3), padding=(1, 1, 1)),
            nn.BatchNorm3d(32), nn.GELU(),
        )
        self.spectral_pool = nn.AdaptiveAvgPool3d((1, patch_size, patch_size))
        self.spatial_pool  = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Sequential(
            nn.Linear(32, out_dim), nn.LayerNorm(out_dim), nn.GELU()
        )

    def forward(self, x):
        x = self.conv3d(x.unsqueeze(1))       # (B, 32, n_bands, S, S)
        x = self.spectral_pool(x).squeeze(2)  # (B, 32, S, S)
        x = self.spatial_pool(x).view(x.size(0), -1)  # (B, 32)
        return self.proj(x)                   # (B, out_dim)


class DualScaleSpatialContext(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Both branches output full SPAT_DIM
        self.local_branch  = SpatialBranch(cfg.N_BANDS, cfg.SPAT_DIM, cfg.LOCAL_SIZE)
        self.global_branch = SpatialBranch(cfg.N_BANDS, cfg.SPAT_DIM, cfg.GLOBAL_SIZE)
        # Scalar gate — single bottleneck, interpretable per sample
        self.gate = nn.Sequential(
            nn.Linear(cfg.SPAT_DIM * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()   # (B, 1): 1 = trust local, 0 = trust global
        )

    def forward(self, local_patch, global_patch):
        F_l = self.local_branch(local_patch)             # (B, SPAT_DIM)
        F_g = self.global_branch(global_patch)           # (B, SPAT_DIM)
        g   = self.gate(torch.cat([F_l, F_g], dim=-1))  # (B, 1)
        return g * F_l + (1 - g) * F_g, g               # (B, SPAT_DIM), (B, 1)

In [ ]:
# ── Full Hy-BiSSM Model ───────────────────────────────────────────────────────

class HyBiSSM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.spectral_encoder = SpectralEncoder(cfg)
        self.spatial_context  = DualScaleSpatialContext(cfg)

        self.classifier = nn.Sequential(
            nn.Linear(cfg.SPEC_DIM + cfg.SPAT_DIM, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, cfg.N_CLASSES)
        )

    def forward(self, local_patch, global_patch):
        B, C, H, W  = local_patch.shape
        center_spec = local_patch[:, :, H // 2, W // 2]     # (B, n_bands)
        spec_feat   = self.spectral_encoder(center_spec)    # (B, SPEC_DIM)
        spat_feat, gate_val = self.spatial_context(local_patch, global_patch)
        logits = self.classifier(torch.cat([spec_feat, spat_feat], dim=-1))
        return logits, spec_feat, gate_val


# Build model
_model = HyBiSSM(CFG).to(DEVICE)
if N_GPUS > 1:
    model = nn.DataParallel(_model)
    print(f'DataParallel across {N_GPUS} GPUs')
else:
    model = _model

total_params     = sum(p.numel() for p in _model.parameters())
trainable_params = sum(p.numel() for p in _model.parameters() if p.requires_grad)
print(f'Parameters : {total_params:,}  trainable: {trainable_params:,}')

## 5. Loss & Optimiser

In [ ]:
# Cross-entropy with label smoothing — the only active loss term
criterion = nn.CrossEntropyLoss(label_smoothing=0.05).to(DEVICE)

# Access underlying model params regardless of DataParallel wrapper
raw_model = model.module if N_GPUS > 1 else model
optimizer = torch.optim.AdamW(raw_model.parameters(),
                              lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)

## 6. Training Loop

In [ ]:
def run_epoch(loader, model, criterion, optimizer=None, device=DEVICE):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for local_p, global_p, lbls in loader:
            local_p  = local_p.to(device)
            global_p = global_p.to(device)
            lbls     = lbls.to(device)

            logits, spec_feat, gate_val = model(local_p, global_p)
            loss = criterion(logits, lbls)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=1.0)
                optimizer.step()

            n = len(lbls)
            total_loss  += loss.item() * n
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())

    oa    = accuracy_score(all_labels, all_preds)
    kappa = cohen_kappa_score(all_labels, all_preds)
    cm    = confusion_matrix(all_labels, all_preds,
                             labels=list(range(CFG.N_CLASSES)))
    aa    = (cm.diagonal() / (cm.sum(axis=1) + 1e-8)).mean()
    N     = len(all_labels)
    return total_loss / N, oa, aa, kappa


history = {k: [] for k in
           ['train_loss', 'val_loss', 'train_oa', 'val_oa',
            'train_aa',   'val_aa',   'train_kappa', 'val_kappa']}

best_val_oa = 0.0
no_improve  = 0
start_epoch = 1

print(f'{"Epoch":>6}  {"Tr.Loss":>8}  {"Tr.OA":>7}  '
      f'{"Val.Loss":>8}  {"Val.OA":>7}  {"Val.AA":>7}  {"Val.K":>7}  {"LR":>9}')
print('-' * 80)

t0 = time.time()
for epoch in range(start_epoch, CFG.EPOCHS + 1):
    tr = run_epoch(train_loader, model, criterion, optimizer)
    vl = run_epoch(val_loader,   model, criterion)
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    for key, val in zip(
        ['train_loss', 'val_loss', 'train_oa', 'val_oa',
         'train_aa',   'val_aa',   'train_kappa', 'val_kappa'],
        [tr[0], vl[0], tr[1], vl[1], tr[2], vl[2], tr[3], vl[3]]):
        history[key].append(val)

    if epoch % 5 == 0 or epoch == 1:
        elapsed = (time.time() - t0) / 60
        print(f'{epoch:>6}  {tr[0]:>8.4f}  {tr[1]:>7.4f}  '
              f'{vl[0]:>8.4f}  {vl[1]:>7.4f}  {vl[2]:>7.4f}  '
              f'{vl[3]:>7.4f}  {lr:>9.2e}  [{elapsed:.0f}m]')

    if vl[1] > best_val_oa:
        best_val_oa = vl[1]
        no_improve  = 0
        cfg_dict = {k: v for k, v in CFG.__dict__.items()
                    if not k.startswith('__') and not callable(v)}
        torch.save({
            'epoch':           epoch,
            'model_state':     raw_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_oa':          vl[1],
            'val_aa':          vl[2],
            'val_kappa':       vl[3],
            'cfg':             cfg_dict,
        }, CFG.SAVE_PATH)
    else:
        no_improve += 1
        if no_improve >= CFG.PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}.')
            break

total_min = (time.time() - t0) / 60
print(f'\nDone in {total_min:.1f} min  |  Best val OA: {best_val_oa:.4f}')

## 7. Learning Curves

In [ ]:
ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(ep, history['train_loss'], label='Train', color='#534AB7')
axes[0].plot(ep, history['val_loss'],   label='Val',   color='#D85A30')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['train_oa'], label='Train OA', color='#534AB7')
axes[1].plot(ep, history['val_oa'],   label='Val OA',   color='#D85A30')
axes[1].plot(ep, history['val_aa'],   label='Val AA',   color='#1D9E75', linestyle='--')
axes[1].set_title('OA & AA'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim([0, 1]); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history['train_kappa'], label='Train κ', color='#534AB7')
axes[2].plot(ep, history['val_kappa'],   label='Val κ',   color='#D85A30')
axes[2].set_title('Kappa'); axes[2].set_xlabel('Epoch')
axes[2].set_ylim([0, 1]); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Hy-BiSSM — Indian Pines — Learning Curves', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('output/hybissm_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Test Evaluation

In [ ]:
# Load best checkpoint
ckpt = torch.load(CFG.SAVE_PATH, map_location=DEVICE, weights_only=False)
raw_model.load_state_dict(ckpt['model_state'])
print(f"Checkpoint epoch {ckpt['epoch']}  val OA={ckpt['val_oa']:.4f}")

# ── Overall test metrics ───────────────────────────────────────────────────────
_, oa, aa, kap = run_epoch(test_loader, model, criterion)
print(f'\nTest — OA: {oa:.4f}  AA: {aa:.4f}  Kappa: {kap:.4f}')

In [ ]:
# ── Per-class accuracy ─────────────────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for lp, gp, lbls in test_loader:
        logits, _, _ = model(lp.to(DEVICE), gp.to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(lbls.numpy())

print(classification_report(
    all_labels, all_preds,
    target_names=CFG.CLASS_NAMES,
    digits=4))

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────────
cm      = confusion_matrix(all_labels, all_preds, labels=list(range(CFG.N_CLASSES)))
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, data, fmt, title in zip(
    axes, [cm, cm_norm], ['d', '.2f'],
    ['Counts', 'Row-normalised recall']):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=CFG.CLASS_NAMES,
                yticklabels=CFG.CLASS_NAMES,
                ax=ax, linewidths=0.5)
    ax.set_title(title)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    plt.setp(ax.get_yticklabels(), fontsize=8)

plt.suptitle(f'Hy-BiSSM — Indian Pines — OA={oa:.4f}  AA={aa:.4f}  κ={kap:.4f}',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('output/hybissm_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Prediction Map — Full Scene

In [ ]:
def predict_full_scene(model, image, labels, cfg, device=DEVICE, batch_size=512):
    """
    Predicts class labels for every pixel in the scene.
    Returns pred_map : (H, W) int array, 0-indexed class (background pixels → -1).
    """
    H, W, _ = image.shape
    rows, cols = np.where(np.ones((H, W), dtype=bool))
    all_idx    = list(zip(rows.tolist(), cols.tolist()))

    # Use a temporary dataset with ALL pixels (label array set to 1 everywhere
    # so make_splits logic doesn't filter anything; we override the label below)
    dummy_labels = np.ones_like(labels)  # treat all pixels as "labeled"
    ds     = DualPatchDataset(image, dummy_labels, cfg, all_idx, augment=False)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)

    model.eval()
    preds = []
    with torch.no_grad():
        for lp, gp, _ in loader:
            logits, _, _ = model(lp.to(device), gp.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())

    pred_map = np.array(preds).reshape(H, W)
    pred_map[labels == 0] = -1   # mask background
    return pred_map


pred_map = predict_full_scene(model, image, labels, CFG)
gt_map   = labels - 1   # 0-indexed; background → -1
gt_map[labels == 0] = -1

# Colour palette — 16 classes + background (grey)
cmap = plt.get_cmap('tab20', CFG.N_CLASSES)
colors = ['#cccccc'] + [cmap(i) for i in range(CFG.N_CLASSES)]

def to_rgb(class_map, n_classes=16):
    """Convert integer class map (-1=bg, 0..15) to RGB image."""
    H, W = class_map.shape
    rgb  = np.zeros((H, W, 3), dtype=np.float32)
    bg   = plt.matplotlib.colors.to_rgb('#cccccc')
    rgb[class_map == -1] = bg
    for c in range(n_classes):
        rgb[class_map == c] = plt.matplotlib.colors.to_rgb(cmap(c))
    return rgb

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(to_rgb(gt_map));   axes[0].set_title('Ground Truth');  axes[0].axis('off')
axes[1].imshow(to_rgb(pred_map)); axes[1].set_title('Hy-BiSSM Prediction'); axes[1].axis('off')

patches = [mpatches.Patch(color=cmap(i), label=CFG.CLASS_NAMES[i])
           for i in range(CFG.N_CLASSES)]
fig.legend(handles=patches, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.15), fontsize=8)
plt.suptitle(f'Hy-BiSSM — Indian Pines  OA={oa:.4f}', fontsize=13)
plt.tight_layout()
plt.savefig('output/hybissm_pred_map.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nFinal Results — OA: {oa:.4f}  AA: {aa:.4f}  Kappa: {kap:.4f}')